# Core Measures & Enterprise KPI Scorecard
## Domain: Financial & Volumetric Executive Metrics
### Objectives & Methodology:
- Aggregate enterprise-level operational and financial KPIs from transactional facts.
- Calculate total gross sales, physical units sold, average item selling price, and distinct order volume.
- Profile customer purchasing participation by comparing total registered customers against active ordering patrons.
- Consolidate all core business metrics into a unified executive KPI scorecard using `UNION ALL` with type-consistent numeric formatting.


In [ ]:
# Databricks notebook source
# Setup environment and display helper
import os
from pyspark.sql import SparkSession

try:
    display
except NameError:
    def display(df, n=20):
        """Fallback display function for standard PySpark and Python environments."""
        if hasattr(df, "show"):
            df.show(n, truncate=False)
        else:
            print(df)


## 1. Individual Micro-Measures Exploration
Compute standalone baseline metrics for sales revenue, order volumes, and catalog dimensions.


In [ ]:
# Total sales revenue
df_total_sales = spark.sql("SELECT SUM(sales_amount) AS total_sales FROM gold.fact_sales")
display(df_total_sales)


In [ ]:
# Total quantity of items sold
df_total_qty = spark.sql("SELECT SUM(quantity) AS total_quantity FROM gold.fact_sales")
display(df_total_qty)


In [ ]:
# Average selling price
df_avg_price = spark.sql("SELECT AVG(price) AS avg_selling_price FROM gold.fact_sales")
display(df_avg_price)


In [ ]:
# Total distinct order count
df_total_orders = spark.sql("SELECT COUNT(DISTINCT order_number) AS total_orders FROM gold.fact_sales")
display(df_total_orders)


In [ ]:
# Total products, registered customers, and purchasing customers
df_dim_counts = spark.sql("""
SELECT
    (SELECT COUNT(DISTINCT product_key) FROM gold.dim_products) AS total_products,
    (SELECT COUNT(DISTINCT customer_key) FROM gold.dim_customers) AS total_registered_customers,
    (SELECT COUNT(DISTINCT customer_key) FROM gold.fact_sales) AS total_purchasing_customers
""")
display(df_dim_counts)


## 2. Consolidated Executive KPI Scorecard
Generate a unified report showing all key enterprise metrics in a single standardized view.


In [ ]:
# Consolidated executive KPI scorecard
# Explicit DECIMAL casting guarantees consistent schema across UNION branches
df_kpi_scorecard = spark.sql("""
SELECT 'Total Sales' AS measure_name, CAST(SUM(sales_amount) AS DECIMAL(18,2)) AS measure_value FROM gold.fact_sales
UNION ALL
SELECT 'Total Quantity' AS measure_name, CAST(SUM(quantity) AS DECIMAL(18,2)) AS measure_value FROM gold.fact_sales
UNION ALL
SELECT 'Average price' AS measure_name, CAST(AVG(price) AS DECIMAL(18,2)) AS measure_value FROM gold.fact_sales
UNION ALL
SELECT 'Total Nr. Orders' AS measure_name, CAST(COUNT(DISTINCT order_number) AS DECIMAL(18,2)) AS measure_value FROM gold.fact_sales
UNION ALL
SELECT 'Total Nr. Products' AS measure_name, CAST(COUNT(DISTINCT product_key) AS DECIMAL(18,2)) AS measure_value FROM gold.dim_products
UNION ALL
SELECT 'Total Nr. Customers' AS measure_name, CAST(COUNT(DISTINCT customer_key) AS DECIMAL(18,2)) AS measure_value FROM gold.dim_customers
""")
display(df_kpi_scorecard)


## Business Takeaways & Scorecard Insights
- **Revenue Volume**: Total enterprise revenue exceeds 29 million, driven by high-value bike and accessory transactions.
- **Active Customer Engagement**: Over 18,000 active ordering customers demonstrate a high conversion rate from registration to completed purchase.
- **Catalog Health**: 296 distinct active products are generating transactions across 27,000+ customer orders.
